# 04 - Machine Learning Model Experiments

This notebook experiments with different ML models for fraud detection.

## Objectives
- Prepare data for machine learning
- Train and evaluate anomaly detection models
- Train and evaluate clustering models
- Compare model performance
- Select best model for production

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, classification_report
import mlflow
import mlflow.sklearn

import sys
sys.path.insert(0, '..')

from src.etl.extractors import extract_all
from src.etl.transformers import transform_all
from src.features.aggregations import create_fraud_detection_dataset
from src.config.settings import MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME

pd.set_option('display.max_columns', None)
%matplotlib inline

In [ ]:
# Setup MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

In [ ]:
# Load and transform data
raw_data = extract_all()
data = transform_all(raw_data)

orders = data['orders']
drivers = data['drivers']
customers = data['customers']

print("Data loaded successfully!")

## 1. Prepare Dataset for ML

In [ ]:
# Create consolidated dataset
ml_data = create_fraud_detection_dataset(orders, drivers, customers)
print(f"ML Dataset shape: {ml_data.shape}")
ml_data.head()

In [ ]:
# Select features for modeling
feature_cols = [
    'order_amount', 'items_delivered', 'items_missing', 'total_items',
    'missing_rate', 'is_high_value', 'is_weekend',
    'driver_age', 'trips', 'driver_missing_rate', 'driver_total_orders',
    'customer_age', 'customer_missing_rate', 'customer_total_orders'
]

# Filter columns that exist
available_cols = [col for col in feature_cols if col in ml_data.columns]
print(f"Using {len(available_cols)} features: {available_cols}")

In [ ]:
# Prepare feature matrix
X = ml_data[available_cols].copy()

# Handle missing values
X = X.fillna(X.median())

# Convert boolean to int
for col in X.columns:
    if X[col].dtype == 'bool':
        X[col] = X[col].astype(int)

print(f"Feature matrix shape: {X.shape}")
X.describe()

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Features scaled successfully!")

## 2. Isolation Forest (Anomaly Detection)

In [ ]:
# Train Isolation Forest
with mlflow.start_run(run_name="isolation_forest"):
    # Model parameters
    contamination = 0.1  # Expected proportion of outliers
    n_estimators = 100
    
    mlflow.log_params({
        "model_type": "IsolationForest",
        "contamination": contamination,
        "n_estimators": n_estimators,
        "n_features": len(available_cols)
    })
    
    # Train model
    iso_forest = IsolationForest(
        contamination=contamination,
        n_estimators=n_estimators,
        random_state=42,
        n_jobs=-1
    )
    
    predictions = iso_forest.fit_predict(X_scaled)
    anomaly_scores = iso_forest.score_samples(X_scaled)
    
    # Results
    n_anomalies = (predictions == -1).sum()
    anomaly_rate = n_anomalies / len(predictions) * 100
    
    mlflow.log_metrics({
        "n_anomalies": n_anomalies,
        "anomaly_rate": anomaly_rate
    })
    
    mlflow.sklearn.log_model(iso_forest, "model")
    
    print(f"Anomalies detected: {n_anomalies} ({anomaly_rate:.2f}%)")

In [ ]:
# Add predictions to data
ml_data['iso_forest_pred'] = predictions
ml_data['iso_forest_score'] = anomaly_scores

# Analyze anomalies
anomalies_if = ml_data[ml_data['iso_forest_pred'] == -1]
print(f"\nAnomaly characteristics:")
print(f"  Avg missing rate: {anomalies_if['missing_rate'].mean():.2f}%")
print(f"  Avg order amount: ${anomalies_if['order_amount'].mean():.2f}")

In [ ]:
# Visualize anomaly scores
fig = px.histogram(
    ml_data,
    x='iso_forest_score',
    color=ml_data['iso_forest_pred'].map({1: 'Normal', -1: 'Anomaly'}),
    nbins=50,
    title='Isolation Forest Anomaly Scores',
    labels={'iso_forest_score': 'Anomaly Score', 'color': 'Classification'}
)
fig.show()

## 3. Local Outlier Factor (LOF)

In [ ]:
# Train LOF
with mlflow.start_run(run_name="local_outlier_factor"):
    n_neighbors = 20
    contamination = 0.1
    
    mlflow.log_params({
        "model_type": "LocalOutlierFactor",
        "n_neighbors": n_neighbors,
        "contamination": contamination
    })
    
    lof = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=contamination,
        novelty=False,
        n_jobs=-1
    )
    
    lof_predictions = lof.fit_predict(X_scaled)
    lof_scores = lof.negative_outlier_factor_
    
    n_anomalies = (lof_predictions == -1).sum()
    anomaly_rate = n_anomalies / len(lof_predictions) * 100
    
    mlflow.log_metrics({
        "n_anomalies": n_anomalies,
        "anomaly_rate": anomaly_rate
    })
    
    print(f"LOF Anomalies detected: {n_anomalies} ({anomaly_rate:.2f}%)")

In [ ]:
# Add LOF predictions
ml_data['lof_pred'] = lof_predictions
ml_data['lof_score'] = lof_scores

## 4. K-Means Clustering

In [ ]:
# Find optimal k using elbow method
inertias = []
silhouettes = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, kmeans.labels_))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(k_range, silhouettes, 'ro-')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')

plt.tight_layout()
plt.show()

optimal_k = k_range[np.argmax(silhouettes)]
print(f"Optimal k based on silhouette: {optimal_k}")

In [ ]:
# Train K-Means with optimal k
with mlflow.start_run(run_name="kmeans_clustering"):
    mlflow.log_params({
        "model_type": "KMeans",
        "n_clusters": optimal_k
    })
    
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    
    silhouette = silhouette_score(X_scaled, cluster_labels)
    
    mlflow.log_metrics({
        "silhouette_score": silhouette,
        "inertia": kmeans.inertia_
    })
    
    mlflow.sklearn.log_model(kmeans, "model")
    
    print(f"K-Means Silhouette Score: {silhouette:.4f}")

In [ ]:
# Add cluster labels
ml_data['cluster'] = cluster_labels

# Analyze clusters
cluster_analysis = ml_data.groupby('cluster').agg({
    'order_id': 'count',
    'missing_rate': 'mean',
    'order_amount': 'mean',
    'has_missing': 'mean'
}).round(2)
cluster_analysis.columns = ['count', 'avg_missing_rate', 'avg_order_amount', 'pct_with_missing']
cluster_analysis

In [ ]:
# Visualize clusters
fig = px.scatter(
    ml_data,
    x='order_amount',
    y='missing_rate',
    color='cluster',
    title='K-Means Clusters: Order Amount vs Missing Rate',
    labels={'order_amount': 'Order Amount ($)', 'missing_rate': 'Missing Rate (%)'}
)
fig.show()

## 5. DBSCAN Clustering

In [ ]:
# Train DBSCAN
with mlflow.start_run(run_name="dbscan_clustering"):
    eps = 0.5
    min_samples = 10
    
    mlflow.log_params({
        "model_type": "DBSCAN",
        "eps": eps,
        "min_samples": min_samples
    })
    
    dbscan = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
    dbscan_labels = dbscan.fit_predict(X_scaled)
    
    n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    n_noise = (dbscan_labels == -1).sum()
    
    mlflow.log_metrics({
        "n_clusters": n_clusters,
        "n_noise_points": n_noise,
        "noise_ratio": n_noise / len(dbscan_labels)
    })
    
    print(f"DBSCAN found {n_clusters} clusters")
    print(f"Noise points (potential anomalies): {n_noise} ({n_noise/len(dbscan_labels)*100:.2f}%)")

In [ ]:
# Add DBSCAN labels
ml_data['dbscan_cluster'] = dbscan_labels

# Analyze noise points
noise_points = ml_data[ml_data['dbscan_cluster'] == -1]
print(f"\nNoise point characteristics:")
print(f"  Avg missing rate: {noise_points['missing_rate'].mean():.2f}%")
print(f"  Pct with missing items: {noise_points['has_missing'].mean()*100:.2f}%")

## 6. Model Comparison

In [ ]:
# Compare anomaly detection methods
comparison = pd.DataFrame({
    'Method': ['Isolation Forest', 'LOF', 'DBSCAN (noise)'],
    'Anomalies Detected': [
        (ml_data['iso_forest_pred'] == -1).sum(),
        (ml_data['lof_pred'] == -1).sum(),
        (ml_data['dbscan_cluster'] == -1).sum()
    ],
    'Anomaly Rate (%)': [
        (ml_data['iso_forest_pred'] == -1).mean() * 100,
        (ml_data['lof_pred'] == -1).mean() * 100,
        (ml_data['dbscan_cluster'] == -1).mean() * 100
    ]
})
comparison

In [ ]:
# Consensus anomalies (detected by multiple methods)
ml_data['anomaly_votes'] = (
    (ml_data['iso_forest_pred'] == -1).astype(int) +
    (ml_data['lof_pred'] == -1).astype(int) +
    (ml_data['dbscan_cluster'] == -1).astype(int)
)

consensus_anomalies = ml_data[ml_data['anomaly_votes'] >= 2]
print(f"Consensus anomalies (2+ methods agree): {len(consensus_anomalies)}")
print(f"Avg missing rate: {consensus_anomalies['missing_rate'].mean():.2f}%")

## 7. Summary and Recommendations

### Model Performance Summary

| Model | Anomalies | Key Insight |
|-------|-----------|-------------|
| Isolation Forest | [Fill] | [Fill] |
| LOF | [Fill] | [Fill] |
| K-Means | [Fill] clusters | [Fill] |
| DBSCAN | [Fill] noise | [Fill] |

### Recommended Model
[Fill based on results]

### Next Steps
1. Deploy selected model to production
2. Set up real-time scoring pipeline
3. Create monitoring dashboard
4. Implement feedback loop for model improvement